# 3 — High-gap + swear features

Inputs:
- `data/processed/1-imbault_gaps_va.csv`
- `data/processed/2-writestreak_posts_final_clean.csv`

Outputs:
- `data/processed/3-gap_and_swear_features.csv`


In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
from l2affect.utils.config import load_config, resolve

cfg = load_config(REPO_ROOT / ".." / "configs" / "config.yaml")
processed_dir = resolve(cfg["paths"]["processed_dir"])


In [2]:
posts = pd.read_csv(processed_dir / "2-writestreak_posts_final_clean.csv")
gaps = pd.read_csv(processed_dir / "1-imbault_gaps_va.csv")

posts["tokens_list"] = posts["tokens"].map(lambda s: json.loads(s) if isinstance(s,str) else [])
gaps["word"] = gaps["word"].astype(str).str.strip().str.lower()
for c in ["gap_valence","gap_arousal"]:
    gaps[c] = pd.to_numeric(gaps[c], errors="coerce")
gaps = gaps.dropna(subset=["word","gap_valence","gap_arousal"]).copy()
gaps["gap_mag"] = np.sqrt(gaps["gap_valence"]**2 + gaps["gap_arousal"]**2) if "gap_mag" not in gaps.columns else pd.to_numeric(gaps["gap_mag"], errors="coerce")
gaps = gaps.dropna(subset=["gap_mag"]).copy()

gap_mag = dict(zip(gaps["word"], gaps["gap_mag"]))
gap_val = dict(zip(gaps["word"], gaps["gap_valence"]))
gap_aro = dict(zip(gaps["word"], gaps["gap_arousal"]))

## Export high-gap word lists (multiple thresholds)

This cell creates:
- `data/processed/high_gap_words.csv` (master list with flags)
- `data/processed/high_gap_top10pct.csv`
- `data/processed/high_gap_top5pct.csv`
- `data/processed/high_gap_ge_1p5.csv`

Thresholds:
- top 10% by `gap_mag` (q=0.90)
- top 5% by `gap_mag` (q=0.95)
- absolute cutoff `gap_mag >= 1.5`


In [ ]:
# --- High-gap thresholds ---
thr_top10 = float(gaps["gap_mag"].quantile(0.90))
thr_top5  = float(gaps["gap_mag"].quantile(0.95))
thr_abs15 = 1.5

high_gap_words = gaps[["word","gap_mag","gap_valence","gap_arousal"]].copy()
high_gap_words["is_top10pct"] = high_gap_words["gap_mag"] >= thr_top10
high_gap_words["is_top5pct"]  = high_gap_words["gap_mag"] >= thr_top5
high_gap_words["is_ge_1p5"]   = high_gap_words["gap_mag"] >= thr_abs15

def _labels(r):
    labels = []
    if r["is_top5pct"]: labels.append("top5pct")
    if r["is_top10pct"]: labels.append("top10pct")
    if r["is_ge_1p5"]: labels.append("ge_1.5")
    return "|".join(labels) if labels else ""

high_gap_words["threshold_labels"] = high_gap_words.apply(_labels, axis=1)

# --- Write outputs ---
master_out = processed_dir / "3-high_gap_words.csv"
top10_out  = processed_dir / "3-high_gap_top10pct.csv"
top5_out   = processed_dir / "3-high_gap_top5pct.csv"
abs15_out  = processed_dir / "3-high_gap_ge_1p5.csv"

high_gap_words.to_csv(master_out, index=False)
high_gap_words[high_gap_words["is_top10pct"]][["word","gap_mag","gap_valence","gap_arousal"]] \
    .sort_values("gap_mag", ascending=False).to_csv(top10_out, index=False)
high_gap_words[high_gap_words["is_top5pct"]][["word","gap_mag","gap_valence","gap_arousal"]] \
    .sort_values("gap_mag", ascending=False).to_csv(top5_out, index=False)
high_gap_words[high_gap_words["is_ge_1p5"]][["word","gap_mag","gap_valence","gap_arousal"]] \
    .sort_values("gap_mag", ascending=False).to_csv(abs15_out, index=False)

print("Saved:", master_out)
print("Saved:", top10_out, "threshold:", thr_top10)
print("Saved:", top5_out,  "threshold:", thr_top5)
print("Saved:", abs15_out, "threshold:", thr_abs15)

Saved: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\high_gap_words.csv
Saved: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\high_gap_top10pct.csv threshold: 1.838722112406205
Saved: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\high_gap_top5pct.csv threshold: 2.136969107761832
Saved: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\high_gap_ge_1p5.csv threshold: 1.5


## Choose which threshold to use

In [4]:
THRESHOLD_MODE = "top5"   # options: "top10", "top5", "abs15"

thr_map = {"top10": thr_top10, "top5": thr_top5, "abs15": thr_abs15}
thr = thr_map[THRESHOLD_MODE]

high_set = set(gaps.loc[gaps["gap_mag"] >= thr, "word"])

print("Using threshold mode:", THRESHOLD_MODE)
print("gap_mag threshold:", thr)
print("high-gap vocab size:", len(high_set))

Using threshold mode: top5
gap_mag threshold: 2.136969107761832
high-gap vocab size: 101


In [5]:
SWEAR_WORDS = {
    "ass","asshole","bitch","boob","bullshit","butt","cock","damn","dickhead","dork","douche","dummy",
    "faggot","fuck","fucker","homo","horseshit","idiot","jackass","moron","piss","prick","pussy",
    "retard","screw","tit"
}
swear_set = {w.lower() for w in SWEAR_WORDS}

def compute_features(tokens):
    toks = [str(t).lower() for t in tokens]
    denom = max(1, len(toks))

    mags = [gap_mag[t] for t in toks if t in high_set]
    high_gap_count = len(mags)
    high_gap_density = high_gap_count / denom
    high_gap_gapmag_mean = float(np.mean(mags)) if mags else np.nan
    high_gap_gapmag_max = float(np.max(mags)) if mags else np.nan

    vals = [gap_val[t] for t in toks if t in high_set]
    aros = [gap_aro[t] for t in toks if t in high_set]
    high_gap_valence_mean_signed = float(np.mean(vals)) if vals else np.nan
    high_gap_arousal_mean_signed = float(np.mean(aros)) if aros else np.nan

    swear_hits = [t for t in toks if t in swear_set]
    swear_count = len(swear_hits)
    swear_density = swear_count / denom
    swear_present = 1 if swear_count > 0 else 0
    swear_unique_count = len(set(swear_hits))
    swear_types = json.dumps(sorted(set(swear_hits)))

    high_gap_swear_count = sum(1 for t in swear_hits if t in high_set)

    return (
        high_gap_count, high_gap_density, high_gap_gapmag_mean, high_gap_gapmag_max,
        high_gap_valence_mean_signed, high_gap_arousal_mean_signed,
        swear_count, swear_density, swear_present, swear_unique_count, swear_types,
        high_gap_swear_count
    )

feat_cols = [
    "high_gap_count","high_gap_density","high_gap_gapmag_mean","high_gap_gapmag_max",
    "high_gap_valence_mean_signed","high_gap_arousal_mean_signed",
    "swear_count","swear_density","swear_present","swear_unique_count","swear_types",
    "high_gap_swear_count"
]

feat_df = pd.DataFrame(posts["tokens_list"].map(compute_features).tolist(), columns=feat_cols, index=posts.index)
out = posts.drop(columns=["tokens_list"]).join(feat_df)
out.head()

,post_id,post_fullname,author,user_id,created_utc,created_at,title,selftext,subreddit,permalink,...,high_gap_gapmag_mean,high_gap_gapmag_max,high_gap_valence_mean_signed,high_gap_arousal_mean_signed,swear_count,swear_density,swear_present,swear_unique_count,swear_types,high_gap_swear_count
0,kt7xl7,t3_kt7xl7,Namatl,00052db50867a4da,1.610129e+09,2021-01-08 18:04:23+00:00,Streak 1 Writing Prompt,"""I was walking to the store in the evening, th...",WriteStreakEN,/r/WriteStreakEN/comments/kt7xl7/streak_1_writ...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
1,ku0tf4,t3_ku0tf4,Namatl,00052db50867a4da,1.610231e+09,2021-01-09 22:18:54+00:00,Streak 2 Favorite Singer/Band,Mi favorite singer is Michael. I grew up liste...,WriteStreakEN,/r/WriteStreakEN/comments/ku0tf4/streak_2_favo...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
2,j3vkx8,t3_j3vkx8,Stylelike,000a672e1864051f,1.601649e+09,2020-10-02 14:34:06+00:00,Streak 1: My bike.,"In May, I bought my first motorcycle; my first...",WriteStreakEN,/r/WriteStreakEN/comments/j3vkx8/streak_1_my_b...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0
3,j4p7n0,t3_j4p7n0,Stylelike,000a672e1864051f,1.601769e+09,2020-10-03 23:42:25+00:00,Streak 2: First aids,I think that learn first aids is indispensable...,WriteStreakEN,/r/WriteStreakEN/comments/j4p7n0/streak_2_firs...,...,2.326908,2.326908,-1.61,1.68,0,0.0,0,0,[],0
4,j5ct7h,t3_j5ct7h,Stylelike,000a672e1864051f,1.601871e+09,2020-10-05 04:02:59+00:00,Streak 3: What would I do with 10 billion doll...,Well... if I can’t expend it in my family or m...,WriteStreakEN,/r/WriteStreakEN/comments/j5ct7h/streak_3_what...,...,NaN,NaN,NaN,NaN,0,0.0,0,0,[],0


In [6]:
out_path = processed_dir / "3-gap_and_swear_features.csv"
summary_path = processed_dir / "3-gap_and_swear_summary.csv"

out.to_csv(out_path, index=False)

summary = {
    "posts_n": int(len(out)),
    "users_n": int(out["user_id"].nunique()),
    "high_gap_threshold_gapmag": float(thr),
    "posts_with_any_high_gap": int((out["high_gap_count"] > 0).sum()),
    "pct_posts_with_any_high_gap": float((out["high_gap_count"] > 0).mean()),
    "posts_with_any_swear": int((out["swear_present"] == 1).sum()),
    "pct_posts_with_any_swear": float((out["swear_present"] == 1).mean()),
    "total_swear_tokens": int(out["swear_count"].sum()),
}

print("Wrote:", out_path)
summary

Wrote: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\3-gap_and_swear_features.csv


{'posts_n': 16471,
 'users_n': 869,
 'high_gap_threshold_gapmag': 2.136969107761832,
 'posts_with_any_high_gap': 9168,
 'pct_posts_with_any_high_gap': 0.5566146560621699,
 'posts_with_any_swear': 214,
 'pct_posts_with_any_swear': 0.012992532329548905,
 'total_swear_tokens': 281}